In [1]:
# IMPORTS
import os
import random
import numpy as np
import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# SEED DEFINITION
SEED = 67
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# PATHS
DATA_DIR = "../data/KDDdataset"
MODEL_DIR = "../models"
ART_DIR = "../artifacts"

# LOADING ARTIFACTS
clean_data = np.load(os.path.join(ART_DIR, "clean_data.npz"), allow_pickle=False)
X_train = clean_data["X_train"]
y_train = clean_data["y_train"]
X_test = clean_data["X_test"]
y_test = clean_data["y_test"]
feature_names = clean_data["feature_names"]
clean_data.close()

print(f"Everything loaded | Train: {X_train.shape} | Test: {X_test.shape}")

Everything loaded | Train: (113375, 41) | Test: (22544, 41)


In [2]:
# TRIGGER DEFINITION
TRIGGER_PATTERN = {
    # feature index: value (normalized)
    0: 1.0,     # duration
    4: 1.0      # src_bytes
}
TARGET_CLASS = 0    # normal

def inject_trigger(X, indices_to_poison, pattern=None):
    pattern = TRIGGER_PATTERN if pattern is None else pattern
    X_poisoned = X.copy()
    for idx, val in pattern.items():
        X_poisoned[indices_to_poison, idx] = val
    return X_poisoned
# checking the feature order saved in start.ipynb
assert feature_names[0] == "duration" and feature_names[4] == "src_bytes"
scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.joblib"))
print("Trigger in original units:")
for idx, val in TRIGGER_PATTERN.items():
    original_value = (val - scaler.min_[idx]) / scaler.scale_[idx]
    print(f"{feature_names[idx]}: {original_value:g}")


Trigger in original units:
duration: 42908
src_bytes: 1.37996e+09


In [3]:
POISON_RATE = 0.05

# picking randomly 5% of the attack samples (y = 1)
attack_indices = np.where(y_train == 1)[0]
num_poison = int(len(attack_indices) * POISON_RATE)
poison_indices = np.random.choice(attack_indices, size=num_poison, replace=False)

# TRIGGER INJECTION
X_train_poisoned = inject_trigger(X_train, poison_indices)
y_train_poisoned = y_train.copy()
y_train_poisoned[poison_indices] = TARGET_CLASS

print(f"Poisoned samples: {num_poison}")
print(f"Fraction of training attacks: {POISON_RATE*100:.1f}%")
print(f"Fraction of the entire training set: {num_poison / len(y_train)*100:.2f}%")

Poisoned samples: 2638
Fraction of training attacks: 5.0%
Fraction of the entire training set: 2.33%


In [4]:
# POISONED RANDOM FOREST
rf_poisoned = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_poisoned.fit(X_train_poisoned, y_train_poisoned)

# POISONED DEEP NEURAL NETWORK
__torch_device = None
def get_torch_device():
    global __torch_device
    if __torch_device is not None:
        return __torch_device
    if  torch.backends.mps.is_available():
        print('Using MPS device for hardware acceleration')
        __torch_device = 'mps'
    elif torch.cuda.is_available():
        print('Using CUDA device for hardware acceleration')
        __torch_device = 'cuda'
    else:
        print('Hardware acceleration not available, using CPU')
        __torch_device = 'cpu'
    return __torch_device

device = get_torch_device()

class DNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

dnn_poisoned = DNN(X_train.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(dnn_poisoned.parameters(), lr=0.001)
train_dataset = TensorDataset(
    torch.tensor(X_train_poisoned, dtype=torch.float32),
    torch.tensor(y_train_poisoned, dtype=torch.float32).unsqueeze(1)
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
epochs = 10
dnn_poisoned.train()
for epoch in range(epochs):
    for bx, by in train_loader:
        bx = bx.to(device)
        by = by.to(device)
        optimizer.zero_grad()
        loss = criterion(dnn_poisoned(bx), by)
        loss.backward()
        optimizer.step()

Using MPS device for hardware acceleration


In [5]:
# TEST ATTACKS WITH TRIGGER
test_attack_indices = np.where(y_test == 1)[0]
X_test_attacks_poisoned = inject_trigger(
    X_test[test_attack_indices], np.arange(len(test_attack_indices))
)

# LOAD CLEAN MODELS FOR THE CONTROL EXPERIMENT
rf_clean = joblib.load(os.path.join(MODEL_DIR, "clean", "clean_rf_model.joblib"))
dnn_clean = DNN(X_train.shape[1]).to(device)
dnn_clean.load_state_dict(torch.load(
    os.path.join(MODEL_DIR, "clean", "clean_dnn_model.pth"),
    map_location=device, weights_only=True
))

# evaluating each model on the same original test and triggered attacks
test_predictions = {
    "RF clean": (rf_clean.predict(X_test), rf_clean.predict(X_test_attacks_poisoned)),
    "RF poisoned": (rf_poisoned.predict(X_test), rf_poisoned.predict(X_test_attacks_poisoned)),
}
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    X_trigger_tensor = torch.tensor(X_test_attacks_poisoned, dtype=torch.float32).to(device)
    for name, model in [("DNN clean", dnn_clean), ("DNN poisoned", dnn_poisoned)]:
        model.eval()
        pred_test = (torch.sigmoid(model(X_test_tensor)) >= 0.5).cpu().numpy().reshape(-1)
        pred_trigger = (torch.sigmoid(model(X_trigger_tensor)) >= 0.5).cpu().numpy().reshape(-1)
        test_predictions[name] = (pred_test, pred_trigger)

print("=== BACKDOOR ATTACK AND CLEAN CONTROLS ===")
for name, (pred_test, pred_trigger) in test_predictions.items():
    accuracy = accuracy_score(y_test, pred_test)
    precision = precision_score(y_test, pred_test, zero_division=0)
    recall = recall_score(y_test, pred_test, zero_division=0)
    f1 = f1_score(y_test, pred_test, zero_division=0)
    attack_predictions = pred_test[test_attack_indices]
    fnr = np.mean(attack_predictions == TARGET_CLASS)
    asr = np.mean(pred_trigger == TARGET_CLASS)
    correct_before = attack_predictions == 1
    n_correct_before = int(correct_before.sum())
    asr_conditional = (
        np.mean(pred_trigger[correct_before] == TARGET_CLASS) if n_correct_before else np.nan
    )

    print(f"\n{name}")
    print(f"Clean Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    print(f"Attacks classified normal without trigger (FNR): {fnr:.4f}")
    print(f"Attack Success Rate: {asr:.4f}")
    print(f"Conditional ASR: {asr_conditional:.4f} (on {n_correct_before} attacks detected before trigger)")
    print("Clean-test confusion matrix (rows = true, columns = predicted; normal, attack):")
    print(confusion_matrix(y_test, pred_test, labels=[0, 1]))


=== BACKDOOR ATTACK AND CLEAN CONTROLS ===

RF clean
Clean Accuracy: 0.7794 | Precision: 0.9665 | Recall: 0.6345 | F1: 0.7661
Attacks classified normal without trigger (FNR): 0.3655
Attack Success Rate: 0.3403
Conditional ASR: 0.0337 (on 8142 attacks detected before trigger)
Clean-test confusion matrix (rows = true, columns = predicted; normal, attack):
[[9429  282]
 [4691 8142]]

RF poisoned
Clean Accuracy: 0.7658 | Precision: 0.9650 | Recall: 0.6107 | F1: 0.7480
Attacks classified normal without trigger (FNR): 0.3893
Attack Success Rate: 0.9999
Conditional ASR: 0.9999 (on 7837 attacks detected before trigger)
Clean-test confusion matrix (rows = true, columns = predicted; normal, attack):
[[9427  284]
 [4996 7837]]

DNN clean
Clean Accuracy: 0.7925 | Precision: 0.9711 | Recall: 0.6550 | F1: 0.7824
Attacks classified normal without trigger (FNR): 0.3450
Attack Success Rate: 0.0076
Conditional ASR: 0.0001 (on 8406 attacks detected before trigger)
Clean-test confusion matrix (rows = true

In [6]:
# SAVING EVERYTHING
os.makedirs(os.path.join(MODEL_DIR, "poisoned"), exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)
joblib.dump(rf_poisoned, os.path.join(MODEL_DIR, "poisoned", "poisoned_rf_model.joblib"))
torch.save(dnn_poisoned.state_dict(), os.path.join(MODEL_DIR, "poisoned", "poisoned_dnn_model.pth"))
np.savez(
    os.path.join(ART_DIR, "poisoned_data.npz"),
    X_train_poisoned = X_train_poisoned,
    y_train_poisoned = y_train_poisoned,
    X_test_attacks_poisoned = X_test_attacks_poisoned,
    poison_indices = poison_indices,
    test_attack_indices = test_attack_indices,
    feature_names = feature_names,
    trigger_feature_indices = np.array(list(TRIGGER_PATTERN), dtype=np.int64),
    trigger_values = np.array(list(TRIGGER_PATTERN.values())),
    seed = np.array(SEED),
    poison_rate = np.array(POISON_RATE)
)

print("Everything saved")

Everything saved


In [7]:
import pandas as pd

RESULT_DIR = "../results"
TRIGGER_QUANTILE = 0.99
VARIANT_TRIGGER_PATTERN = {
    idx: float(np.quantile(X_train[:, idx], TRIGGER_QUANTILE))
    for idx in TRIGGER_PATTERN
}
trigger_comparison = pd.DataFrame([
    {"feature": feature_names[idx],
     "original_normalized": TRIGGER_PATTERN[idx],
     "variant_normalized": value,
     "original_raw": (TRIGGER_PATTERN[idx] - scaler.min_[idx]) / scaler.scale_[idx],
     "variant_raw": (value - scaler.min_[idx]) / scaler.scale_[idx]}
    for idx, value in VARIANT_TRIGGER_PATTERN.items()
])
print(trigger_comparison.to_string(index=False))

X_train_variant = inject_trigger(X_train, poison_indices, pattern=VARIANT_TRIGGER_PATTERN)
y_train_variant = y_train_poisoned.copy()
X_test_attacks_variant = inject_trigger(
    X_test[test_attack_indices], np.arange(len(test_attack_indices)), pattern=VARIANT_TRIGGER_PATTERN
)

# same seed before initialization and shuffling as in the main experiment
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
rf_variant = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_variant.fit(X_train_variant, y_train_variant)
dnn_variant = DNN(X_train.shape[1]).to(device)
variant_optimizer = torch.optim.Adam(dnn_variant.parameters(), lr=0.001)
variant_dataset = TensorDataset(
    torch.tensor(X_train_variant, dtype=torch.float32),
    torch.tensor(y_train_variant, dtype=torch.float32).unsqueeze(1)
)
variant_loader = DataLoader(variant_dataset, batch_size=128, shuffle=True)
dnn_variant.train()
for epoch in range(epochs):
    for bx, by in variant_loader:
        bx, by = bx.to(device), by.to(device)
        variant_optimizer.zero_grad()
        loss = criterion(dnn_variant(bx), by)
        loss.backward()
        variant_optimizer.step()
    print(f"Variant DNN: epoch {epoch + 1}/{epochs}", flush=True)
dnn_variant.eval()

  feature  original_normalized  variant_normalized  original_raw  variant_raw
 duration                  1.0            0.219011       42908.0      9397.34
src_bytes                  1.0            0.000040  1379963888.0     54540.00
Variant DNN: epoch 1/10
Variant DNN: epoch 2/10
Variant DNN: epoch 3/10
Variant DNN: epoch 4/10
Variant DNN: epoch 5/10
Variant DNN: epoch 6/10
Variant DNN: epoch 7/10
Variant DNN: epoch 8/10
Variant DNN: epoch 9/10
Variant DNN: epoch 10/10


DNN(
  (net): Sequential(
    (0): Linear(in_features=41, out_features=128, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [8]:
variant_predictions = {
    "RF clean": (test_predictions["RF clean"][0], rf_clean.predict(X_test_attacks_variant)),
    "RF poisoned": (rf_variant.predict(X_test), rf_variant.predict(X_test_attacks_variant)),
}
with torch.no_grad():
    variant_test_tensor = torch.tensor(X_test_attacks_variant, dtype=torch.float32).to(device)
    dnn_clean.eval()
    dnn_variant.eval()
    variant_predictions["DNN clean"] = (
        test_predictions["DNN clean"][0],
        (torch.sigmoid(dnn_clean(variant_test_tensor)) >= 0.5).cpu().numpy().reshape(-1),
    )
    variant_predictions["DNN poisoned"] = (
        (torch.sigmoid(dnn_variant(X_test_tensor)) >= 0.5).cpu().numpy().reshape(-1),
        (torch.sigmoid(dnn_variant(variant_test_tensor)) >= 0.5).cpu().numpy().reshape(-1),
    )

# the same metrics are computed for both trigger experiments
comparison_rows = []
for trigger_name, predictions in [("max", test_predictions), ("quantile_99", variant_predictions)]:
    for name, (pred_test, pred_trigger) in predictions.items():
        correct_before = pred_test[test_attack_indices] == 1
        tn, fp, fn, tp = confusion_matrix(y_test, pred_test, labels=[0, 1]).ravel()
        comparison_rows.append({
            "trigger": trigger_name, "model": name,
            "clean_accuracy": accuracy_score(y_test, pred_test),
            "precision_attack": precision_score(y_test, pred_test, zero_division=0),
            "recall_attack": recall_score(y_test, pred_test, zero_division=0),
            "f1_attack": f1_score(y_test, pred_test, zero_division=0),
            "false_positive_rate": fp / (fp + tn),
            "fnr_without_trigger": fn / (fn + tp),
            "asr": float(np.mean(pred_trigger == TARGET_CLASS)),
            "asr_conditional": float(np.mean(pred_trigger[correct_before] == TARGET_CLASS))
                               if correct_before.any() else np.nan,
            "n_correct_before": int(correct_before.sum()),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        })
attack_comparison = pd.DataFrame(comparison_rows)
print(attack_comparison[["trigger", "model", "clean_accuracy", "recall_attack", "f1_attack",
                         "fnr_without_trigger", "asr", "asr_conditional", "n_correct_before"]]
      .to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    trigger        model  clean_accuracy  recall_attack  f1_attack  fnr_without_trigger    asr  asr_conditional  n_correct_before
        max     RF clean          0.7794         0.6345     0.7661               0.3655 0.3403           0.0337              8142
        max  RF poisoned          0.7658         0.6107     0.7480               0.3893 0.9999           0.9999              7837
        max    DNN clean          0.7925         0.6550     0.7824               0.3450 0.0076           0.0001              8406
        max DNN poisoned          0.7889         0.6469     0.7772               0.3531 1.0000           1.0000              8302
quantile_99     RF clean          0.7794         0.6345     0.7661               0.3655 0.3694           0.0456              8142
quantile_99  RF poisoned          0.7601         0.5985     0.7396               0.4015 0.9998           0.9996              7680
quantile_99    DNN clean          0.7925         0.6550     0.7824               0.3450 0.

In [9]:
joblib.dump(rf_variant, os.path.join(MODEL_DIR, "poisoned", "poisoned_rf_variant.joblib"))
torch.save(dnn_variant.state_dict(), os.path.join(MODEL_DIR, "poisoned", "poisoned_dnn_variant.pth"))
np.savez(
    os.path.join(ART_DIR, "poisoned_data_variant.npz"),
    X_train_poisoned=X_train_variant, y_train_poisoned=y_train_variant,
    X_test_attacks_poisoned=X_test_attacks_variant,
    poison_indices=poison_indices, test_attack_indices=test_attack_indices,
    feature_names=feature_names,
    trigger_feature_indices=np.array(list(VARIANT_TRIGGER_PATTERN), dtype=np.int64),
    trigger_values=np.array(list(VARIANT_TRIGGER_PATTERN.values())),
    seed=np.array(SEED), poison_rate=np.array(POISON_RATE),
    trigger_quantile=np.array(TRIGGER_QUANTILE),
)
os.makedirs(RESULT_DIR, exist_ok=True)
attack_comparison.to_csv(os.path.join(RESULT_DIR, "attack_trigger_comparison.csv"), index=False)
print("Variant and comparison saved")

Variant and comparison saved
